In [ ]:
# =============================================================================
# jobs/20_mp_briefing_drafts.py
# Draft a short flood briefing for each MP whose constituency is forecast at
# risk. DRAFTS ONLY. Nothing in this notebook sends anything.
#
# What this script does:
#   1. Reads constituency_flood_situation (19): one row per constituency.
#   2. Picks constituencies whose forecast reaches BRIEFING_MIN_RISK_RANK
#      (config.py) on any of the five days and that have a matched MP.
#   3. Adds the impact and likelihood behind the worst risk, from
#      fgs_constituency_mp_contacts (14), and the flood warning and flood alert
#      areas that sit under the worst-risk polygons (fgs_ea_area_intersections,
#      linked to the constituency by ea_area_constituency_lookup). These are
#      potential warnings and alerts, places where one could be issued.
#   4. Writes a subject and a plain-text body per MP to mp_briefing_drafts, with
#      status = 'draft', for a person to review, edit and send by their own
#      route.
#
# WHY DRAFTS: a message to an MP is a policy decision, and a wrong or badly
# pitched one cannot be recalled. The wording here is deliberately factual: what
# the statement says, which days, which sources, whether it has risen. It adds no
# advice and makes no promise. The sender and sign-off are left as placeholders.
#
# IDEMPOTENT. Re-running for the same statement replaces drafts still marked
# 'draft' and leaves any row a person has moved on (status other than 'draft')
# exactly as it is. Older statements' drafts stay as history; select the latest
# statement_id to see the current batch.
#
# Exit values: "success" (drafts written, possibly none).
# =============================================================================

import os
import sys

# Resolve the project root from this notebook's own location, so nothing
# depends on one person's workspace folder.
_nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
if not _nb_path.startswith("/Workspace"):
    _nb_path = "/Workspace" + _nb_path
PROJECT_ROOT = os.path.dirname(os.path.dirname(_nb_path))
sys.path.insert(0, PROJECT_ROOT)

from config import (
    TBL_STATEMENTS, TBL_FGS_INTERSECT_LOG, TBL_CONST_SITUATION, TBL_FGS_CONTACTS,
    TBL_MP_BRIEFING_DRAFTS, BRIEFING_MIN_RISK_RANK, FGS_GEOMETRY_SOURCE,
    TBL_FGS_EA_INTERSECT, TBL_EA_CONST_LOOKUP, WARNING_CONTACT_MIN_PCT
)
from utils.helpers import (
    get_spark, utc_now, table_exists, latest_complete_statements,
    risk_rank_sql, source_predicate, areas_under_forecast_sentence, BRIEFING_DRAFT_SCHEMA
)
from utils.briefing import build_draft as build_briefing, day_dates, optional_bool


# =============================================================================
# SETUP
# =============================================================================

spark   = get_spark()
now_iso = utc_now().isoformat()

for table in (TBL_STATEMENTS, TBL_FGS_INTERSECT_LOG, TBL_CONST_SITUATION, TBL_FGS_CONTACTS):
    if not table_exists(spark, table):
        raise Exception(f"{table} does not exist. Run 07, 14 and 19 first.")

complete = latest_complete_statements(
    spark, TBL_STATEMENTS, TBL_FGS_INTERSECT_LOG, FGS_GEOMETRY_SOURCE, n=1
)
if not complete:
    raise Exception(
        f"No statement has up-to-date intersections for geometry source "
        f"'{FGS_GEOMETRY_SOURCE}'. Run 07_compute_intersections first."
    )
statement_id = int(complete[0]["statement_id"])

held = {r["fgs_statement_id"] for r in
        spark.table(TBL_CONST_SITUATION).select("fgs_statement_id").distinct().collect()}
if held != {statement_id}:
    raise Exception(
        f"constituency_flood_situation holds statement {sorted(held)} but the latest "
        f"complete statement is {statement_id}. Run 14 and 19 first, so no draft is "
        f"written from an out-of-date forecast."
    )

RANK_NAME = {1: "Very Low", 2: "Low", 3: "Medium", 4: "High"}
print(f"Statement {statement_id}; drafting for forecasts of "
      f"{RANK_NAME[BRIEFING_MIN_RISK_RANK]} or higher.")


In [ ]:
# =============================================================================
# CHOOSE WHO GETS A DRAFT, AND GATHER THE DETAIL
# =============================================================================

candidates = (
    spark.table(TBL_CONST_SITUATION)
    .filter(f"fgs_worst_rank >= {BRIEFING_MIN_RISK_RANK}")
    .collect()
)
with_mp    = [c for c in candidates if c["mp_matched"]]
without_mp = [c for c in candidates if not c["mp_matched"]]
print(f"Constituencies at or above the threshold: {len(candidates)}")
if without_mp:
    print(f"  {len(without_mp)} have no matched MP (vacancy, or a name that does not match), "
          f"so no draft: " + ", ".join(sorted(c["constituency_name"] for c in without_mp)[:10]))

# Impact and likelihood behind the worst risk, per constituency, from 14.
ids = ", ".join("'" + c["constituency_id"].replace("'", "''") + "'" for c in with_mp) or "''"
detail_rows = spark.sql(f"""
    SELECT DISTINCT constituency_id, forecast_date, source, risk_rank,
                    impact_label, likelihood_label
    FROM {TBL_FGS_CONTACTS}
    WHERE constituency_id IN ({ids})
""").collect()

detail = {}
for r in detail_rows:
    detail.setdefault(r["constituency_id"], []).append(r)

# Flood warning and alert areas under the forecast polygons, per constituency. An
# area is linked to a constituency when at least WARNING_CONTACT_MIN_PCT of the area
# lies inside it (the rule 13 uses). The draft counts the areas under polygons at
# the constituency's worst risk, for the flood sources behind that risk.
areas_by_const = {}
if table_exists(spark, TBL_FGS_EA_INTERSECT) and table_exists(spark, TBL_EA_CONST_LOOKUP):
    e_in_source = source_predicate(spark, TBL_FGS_EA_INTERSECT, FGS_GEOMETRY_SOURCE, alias="e")
    area_rows = spark.sql(f"""
        SELECT DISTINCT l.constituency_id, e.source,
                        {risk_rank_sql('e.risk_level')} AS risk_rank,
                        e.ea_area_code, LOWER(e.ea_area_type) AS ea_area_type
        FROM {TBL_FGS_EA_INTERSECT} e
        JOIN {TBL_EA_CONST_LOOKUP} l ON l.ea_area_code = e.ea_area_code
        WHERE e.statement_id = {statement_id}
          AND {e_in_source}
          AND l.intersection_pct >= {WARNING_CONTACT_MIN_PCT}
          AND l.constituency_id IN ({ids})
    """).collect()
    for r in area_rows:
        areas_by_const.setdefault(r["constituency_id"], []).append(r)
else:
    print("fgs_ea_area_intersections or ea_area_constituency_lookup is missing, so drafts "
          "say nothing about areas under the forecast.")

# Date of each forecast day. Day 1's date is read from the data; days nothing
# is forecast on are filled in by counting from a known day.
day_date = day_dates(
    (r["day_index"], r["forecast_date"])
    for r in spark.sql(f"SELECT DISTINCT day_index, forecast_date FROM {TBL_FGS_CONTACTS}").collect()
)


In [ ]:
# =============================================================================
# WRITE THE DRAFTS
# =============================================================================

def build_draft(c):
    return build_briefing(
        c, detail.get(c["constituency_id"], []), areas_by_const.get(c["constituency_id"], []),
        day_date, areas_under_forecast_sentence)


DRAFT_SCHEMA = BRIEFING_DRAFT_SCHEMA

# Rows a person has already moved on must not be overwritten.
protected = set()
if table_exists(spark, TBL_MP_BRIEFING_DRAFTS):
    protected = {
        r["constituency_id"] for r in spark.sql(f"""
            SELECT constituency_id FROM {TBL_MP_BRIEFING_DRAFTS}
            WHERE statement_id = {statement_id}
              AND COALESCE(geometry_source, 'risk_areas') = '{FGS_GEOMETRY_SOURCE}'
              AND status <> 'draft'
        """).collect()
    }
    spark.sql(f"""
        DELETE FROM {TBL_MP_BRIEFING_DRAFTS}
        WHERE statement_id = {statement_id}
          AND COALESCE(geometry_source, 'risk_areas') = '{FGS_GEOMETRY_SOURCE}'
          AND status = 'draft'
    """)

rows = []
for c in with_mp:
    if c["constituency_id"] in protected:
        continue
    subject, body = build_draft(c)
    rows.append((
        statement_id, FGS_GEOMETRY_SOURCE, c["constituency_id"], c["constituency_name"],
        int(c["mp_member_id"]), c["mp_name"], c["mp_email"],
        int(c["fgs_worst_rank"]), RANK_NAME[int(c["fgs_worst_rank"])], str(c["fgs_worst_date"])[:10],
        optional_bool(c["fgs_escalated"]), optional_bool(c["has_active_warning"]),
        subject, body, "draft", None, None, now_iso,
    ))

if rows:
    (
        spark.createDataFrame(rows, DRAFT_SCHEMA).write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(TBL_MP_BRIEFING_DRAFTS)
    )
elif not table_exists(spark, TBL_MP_BRIEFING_DRAFTS):
    # A quiet statement has nothing to draft. Create the table empty anyway, so the app
    # (which reports a missing table as a pipeline gap) is not told one is missing.
    (
        spark.createDataFrame([], DRAFT_SCHEMA).write
        .format("delta")
        .mode("append")
        .saveAsTable(TBL_MP_BRIEFING_DRAFTS)
    )

if table_exists(spark, TBL_MP_BRIEFING_DRAFTS):
    spark.sql(f"""
        COMMENT ON TABLE {TBL_MP_BRIEFING_DRAFTS} IS
        'DRAFT briefings for MPs whose constituency is forecast at risk, one row per MP
         per Flood Guidance Statement. Written by 20_mp_briefing_drafts. Nothing is sent
         from here. A person reviews each draft and sends it by their own route, then
         sets status (for example reviewed, sent, rejected) and reviewed_by. Rows whose
         status is not draft are never overwritten. The latest statement_id is the
         current batch.'
    """)

print(f"Drafts written: {len(rows)}   (kept as already reviewed: {len(protected)})")
dbutils.notebook.exit("success")
